# Exploratory Data Analysis - Instacart Dataset

## Bronze Layer - Data Structure and Quality Assessment

This notebook performs data quality assessment on the Bronze layer tables, focusing on structure, robustness, and data integrity.

### Objectives:
* Assess data structure (rows, columns, data types)
* Identify data quality issues (nulls, duplicates, errors)
* Validate categorical values and cardinalities
* Check referential integrity between tables
* Provide recommendations for data cleaning

### Tables Analyzed:
1. **orders** - Customer orders with timing information
2. **products** - Product catalog with categories
3. **order_products_prior** - Products in historical orders
4. **order_products_train** - Products in training set orders
5. **aisles** - Product aisle categories
6. **departments** - Product departments
7. **prices** - Product pricing information

### Analysis Focus:
* Table structures and data types
* Missing values and NULL analysis
* Duplicate detection
* Referential integrity validation
* Data quality summary and recommendations

In [0]:
# PySpark functions
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, count, countDistinct, avg, sum as _sum, round as _round,
    max as _max, min as _min, when, percentile_approx, stddev, variance
)

In [0]:
# Schema configuration
bronze_schema = "big_data.bronze"

# Bronze tables to analyze
bronze_tables = [
    'orders',
    'products', 
    'order_products_prior',
    'order_products_train',
    'aisles',
    'departments',
    'prices'
]

print(f"Source schema: {bronze_schema}")
print(f"Tables to analyze: {len(bronze_tables)}")
print(f"\nTable list:")
for i, table in enumerate(bronze_tables, 1):
    print(f"   {i}. {table}")

## Data Overview and Basic Statistics

In [0]:
print("=" * 80)
print("DATA VOLUME SUMMARY")
print("=" * 80)
print("\n")

# Collect all row counts
table_stats = []
for table_name in bronze_tables:
    full_table = f"{bronze_schema}.{table_name}"
    df = spark.table(full_table)
    row_count = df.count()
    col_count = len(df.columns)
    table_stats.append((table_name, row_count, col_count))

print(f"{'Table Name':<30} {'Rows':>15} {'Columns':>10}")
print("-" * 80)
for table_name, row_count, col_count in sorted(table_stats, key=lambda x: x[1], reverse=True):
    print(f"{table_name:<30} {row_count:>15,} {col_count:>10}")

total_rows = sum([stat[1] for stat in table_stats])
print("=" * 80)
print(f"{'TOTAL':<30} {total_rows:>15,}")
print("=" * 80)

In [0]:
print("=" * 80)
print("TABLE SCHEMAS AND SAMPLE DATA")
print("=" * 80)
print("\nDisplaying schema and sample records for all Bronze tables\n")

for table_name in bronze_tables:
    full_table = f"{bronze_schema}.{table_name}"
    df = spark.table(full_table)
    
    print("\n" + "=" * 80)
    print(f"Table: {table_name.upper()}")
    print("=" * 80)
    
    # Schema
    print("\nSchema:")
    df.printSchema()
    
    # Sample data
    print(f"\nSample Data (first 5 rows):")
    display(df.limit(5))
    print("\n")

## Referential Integrity Analysis

In [0]:
print("=" * 80)
print("REFERENTIAL INTEGRITY ANALYSIS")
print("=" * 80)
print("\nValidating foreign key relationships between tables\n")

# Load tables
orders = spark.table(f"{bronze_schema}.orders")
products = spark.table(f"{bronze_schema}.products")
aisles = spark.table(f"{bronze_schema}.aisles")
departments = spark.table(f"{bronze_schema}.departments")
order_products_prior = spark.table(f"{bronze_schema}.order_products_prior")
order_products_train = spark.table(f"{bronze_schema}.order_products_train")

# 1. Products -> Aisles
print("1. Products -> Aisles Relationship")
product_aisles = products.select("aisle_id").distinct()
aisle_ids = aisles.select("aisle_id").distinct()
unmatched_aisles = product_aisles.subtract(aisle_ids).count()
print(f"   Product aisle_ids not in aisles table: {unmatched_aisles}")
if unmatched_aisles == 0:
    print("   Status: OK - All aisle_ids are valid")
else:
    print("   Status: WARNING - Some aisle_ids are missing")

# 2. Products -> Departments
print("\n2. Products -> Departments Relationship")
product_depts = products.select("department_id").distinct()
dept_ids = departments.select("department_id").distinct()
unmatched_depts = product_depts.subtract(dept_ids).count()
print(f"   Product department_ids not in departments table: {unmatched_depts}")
if unmatched_depts == 0:
    print("   Status: OK - All department_ids are valid")
else:
    print("   Status: WARNING - Some department_ids are missing")

# 3. Order Products Prior -> Orders
print("\n3. Order Products Prior -> Orders Relationship")
prior_order_ids = order_products_prior.select("order_id").distinct()
order_ids = orders.select("order_id").distinct()
unmatched_prior_orders = prior_order_ids.subtract(order_ids).count()
print(f"   Prior order_ids not in orders table: {unmatched_prior_orders}")
if unmatched_prior_orders == 0:
    print("   Status: OK - All order_ids are valid")
else:
    print("   Status: WARNING - Some order_ids are missing")

# 4. Order Products Train -> Orders
print("\n4. Order Products Train -> Orders Relationship")
train_order_ids = order_products_train.select("order_id").distinct()
unmatched_train_orders = train_order_ids.subtract(order_ids).count()
print(f"   Train order_ids not in orders table: {unmatched_train_orders}")
if unmatched_train_orders == 0:
    print("   Status: OK - All order_ids are valid")
else:
    print("   Status: WARNING - Some order_ids are missing")

# 5. Order Products Prior -> Products
print("\n5. Order Products Prior -> Products Relationship")
prior_product_ids = order_products_prior.select("product_id").distinct()
product_ids = products.select("product_id").distinct()
unmatched_prior_products = prior_product_ids.subtract(product_ids).count()
print(f"   Prior product_ids not in products table: {unmatched_prior_products}")
if unmatched_prior_products == 0:
    print("   Status: OK - All product_ids are valid")
else:
    print("   Status: WARNING - Some product_ids are missing")

# 6. Order Products Train -> Products
print("\n6. Order Products Train -> Products Relationship")
train_product_ids = order_products_train.select("product_id").distinct()
unmatched_train_products = train_product_ids.subtract(product_ids).count()
print(f"   Train product_ids not in products table: {unmatched_train_products}")
if unmatched_train_products == 0:
    print("   Status: OK - All product_ids are valid")
else:
    print("   Status: WARNING - Some product_ids are missing")

print("\n" + "=" * 80)
print("REFERENTIAL INTEGRITY CHECK COMPLETE")
print("=" * 80)

## Executive Summary

In [0]:
print("=" * 80)
print("DATA QUALITY AND ROBUSTNESS SUMMARY")
print("=" * 80)
print("\nSummary of data structure, quality issues, and recommendations\n")

# Load tables
orders = spark.table(f"{bronze_schema}.orders")
products = spark.table(f"{bronze_schema}.products")
order_products_prior = spark.table(f"{bronze_schema}.order_products_prior")
order_products_train = spark.table(f"{bronze_schema}.order_products_train")
aisles = spark.table(f"{bronze_schema}.aisles")
departments = spark.table(f"{bronze_schema}.departments")
prices = spark.table(f"{bronze_schema}.prices")

print("=" * 80)
print("1. DATA VOLUME")
print("=" * 80)
print(f"   orders: {orders.count():,} rows, {len(orders.columns)} columns")
print(f"   products: {products.count():,} rows, {len(products.columns)} columns")
print(f"   order_products_prior: {order_products_prior.count():,} rows, {len(order_products_prior.columns)} columns")
print(f"   order_products_train: {order_products_train.count():,} rows, {len(order_products_train.columns)} columns")
print(f"   aisles: {aisles.count():,} rows, {len(aisles.columns)} columns")
print(f"   departments: {departments.count():,} rows, {len(departments.columns)} columns")
print(f"   prices: {prices.count():,} rows, {len(prices.columns)} columns")

print("\n" + "=" * 80)
print("2. DATA QUALITY ISSUES")
print("=" * 80)

# Check for nulls in critical columns
orders_nulls = orders.filter(col("order_id").isNull() | col("user_id").isNull()).count()
print(f"   orders: {orders_nulls} rows with NULL critical fields (order_id, user_id)")

orders_days_nulls = orders.filter(col("days_since_prior_order").isNull()).count()
print(f"   orders: {orders_days_nulls:,} rows with NULL days_since_prior_order (expected for first orders)")

products_nulls = products.filter(col("product_id").isNull() | col("product_name").isNull()).count()
print(f"   products: {products_nulls} rows with NULL critical fields (product_id, product_name)")

# Check duplicates
orders_dups = orders.count() - orders.select("order_id").distinct().count()
print(f"   orders: {orders_dups} duplicate order_id")

products_dups = products.count() - products.select("product_id").distinct().count()
print(f"   products: {products_dups} duplicate product_id")

print("\n" + "=" * 80)
print("3. REFERENTIAL INTEGRITY")
print("=" * 80)

# Products -> Aisles
product_aisles = products.select("aisle_id").distinct()
aisle_ids = aisles.select("aisle_id").distinct()
unmatched_aisles = product_aisles.subtract(aisle_ids).count()
print(f"   Products with invalid aisle_id: {unmatched_aisles}")

# Products -> Departments
product_depts = products.select("department_id").distinct()
dept_ids = departments.select("department_id").distinct()
unmatched_depts = product_depts.subtract(dept_ids).count()
print(f"   Products with invalid department_id: {unmatched_depts}")

# Order Products -> Orders
prior_order_ids = order_products_prior.select("order_id").distinct()
order_ids = orders.select("order_id").distinct()
unmatched_prior_orders = prior_order_ids.subtract(order_ids).count()
print(f"   Order_products_prior with invalid order_id: {unmatched_prior_orders}")

# Order Products -> Products
prior_product_ids = order_products_prior.select("product_id").distinct()
product_ids = products.select("product_id").distinct()
unmatched_prior_products = prior_product_ids.subtract(product_ids).count()
print(f"   Order_products_prior with invalid product_id: {unmatched_prior_products}")

print("\n" + "=" * 80)
print("4. DATA TYPES")
print("=" * 80)
print("   All tables have appropriate data types after Bronze ingestion")
print("   String types: product_name, aisle, department, eval_set")
print("   Numeric types: IDs, order_number, order_dow, order_hour_of_day")
print("   Decimal types: days_since_prior_order, price_usd")
print("   Boolean types: reordered (stored as int: 0/1)")

print("\n" + "=" * 80)
print("5. RECOMMENDATIONS FOR SILVER LAYER")
print("=" * 80)
print("   1. Handle NULL in days_since_prior_order (mark as first order)")
print("   2. Cast all IDs to INT type for consistency")
print("   3. Cast reordered to BOOLEAN type")
print("   4. Validate all foreign key relationships")
print("   5. Join products with aisles and departments for enrichment")
print("   6. Join products with prices (handle missing prices)")
print("   7. Union order_products_prior and train datasets")
print("   8. Remove metadata columns (ingestion_timestamp, source_file)")

print("\n" + "=" * 80)
print("OVERALL DATA QUALITY: GOOD")
print("=" * 80)
print("   No critical issues found")
print("   All referential integrity constraints are valid")
print("   NULL values are within expected ranges")
print("   No duplicates in primary keys")
print("\n")

## Order Products Analysis (Prior and Train)

In [0]:
print("=" * 80)
print("ORDER PRODUCTS TABLES - STRUCTURE AND DATA QUALITY")
print("=" * 80)

order_products_prior = spark.table(f"{bronze_schema}.order_products_prior")
order_products_train = spark.table(f"{bronze_schema}.order_products_train")

# Prior Dataset
print("\n[1] ORDER_PRODUCTS_PRIOR")
print("-" * 80)
print(f"   Total Rows: {order_products_prior.count():,}")
print(f"   Total Columns: {len(order_products_prior.columns)}")
print(f"\n   Schema:")
order_products_prior.printSchema()
print(f"\n   Cardinality:")
print(f"      Unique order_id: {order_products_prior.select('order_id').distinct().count():,}")
print(f"      Unique product_id: {order_products_prior.select('product_id').distinct().count():,}")
print(f"\n   Reordered Distribution:")
reorder_dist_prior = order_products_prior.groupBy("reordered").agg(count("*").alias("count")).orderBy("reordered")
display(reorder_dist_prior)

# Train Dataset
print("\n[2] ORDER_PRODUCTS_TRAIN")
print("-" * 80)
print(f"   Total Rows: {order_products_train.count():,}")
print(f"   Total Columns: {len(order_products_train.columns)}")
print(f"\n   Schema:")
order_products_train.printSchema()
print(f"\n   Cardinality:")
print(f"      Unique order_id: {order_products_train.select('order_id').distinct().count():,}")
print(f"      Unique product_id: {order_products_train.select('product_id').distinct().count():,}")
print(f"\n   Reordered Distribution:")
reorder_dist_train = order_products_train.groupBy("reordered").agg(count("*").alias("count")).orderBy("reordered")
display(reorder_dist_train)

## Products and Categories Analysis

In [0]:
print("=" * 80)
print("PRODUCTS, AISLES, AND DEPARTMENTS - STRUCTURE")
print("=" * 80)

products = spark.table(f"{bronze_schema}.products")
aisles = spark.table(f"{bronze_schema}.aisles")
departments = spark.table(f"{bronze_schema}.departments")

# Products
print("\n[1] PRODUCTS TABLE")
print("-" * 80)
print(f"   Total Rows: {products.count():,}")
print(f"   Total Columns: {len(products.columns)}")
print(f"\n   Schema:")
products.printSchema()
print(f"\n   Cardinality:")
print(f"      Unique product_id: {products.select('product_id').distinct().count():,}")
print(f"      Unique product_name: {products.select('product_name').distinct().count():,}")
print(f"      Unique aisle_id: {products.select('aisle_id').distinct().count():,}")
print(f"      Unique department_id: {products.select('department_id').distinct().count():,}")

# Aisles
print("\n[2] AISLES TABLE (Product Categories)")
print("-" * 80)
print(f"   Total Rows: {aisles.count():,}")
print(f"   Total Columns: {len(aisles.columns)}")
print(f"\n   Schema:")
aisles.printSchema()
print(f"\n   Sample Aisles:")
display(aisles.limit(10))

# Departments
print("\n[3] DEPARTMENTS TABLE (Product Groups)")
print("-" * 80)
print(f"   Total Rows: {departments.count():,}")
print(f"   Total Columns: {len(departments.columns)}")
print(f"\n   Schema:")
departments.printSchema()
print(f"\n   All Departments:")
display(departments.orderBy("department_id"))

In [0]:
print("=" * 80)
print("PRICES TABLE - ANALYSIS")
print("=" * 80)

prices = spark.table(f"{bronze_schema}.prices")

print(f"\nTotal Price Records: {prices.count():,}")
print(f"Unique Products with Prices: {prices.select('product_name').distinct().count():,}")

# Price statistics
print("\nPrice Statistics:")
price_stats = prices.selectExpr("CAST(price_usd AS DOUBLE) as price") \
    .filter(col("price").isNotNull()) \
    .agg(
        _round(_min("price"), 2).alias("min_price"),
        _round(_max("price"), 2).alias("max_price"),
        _round(avg("price"), 2).alias("avg_price"),
        _round(percentile_approx("price", 0.5), 2).alias("median_price"),
        _round(stddev("price"), 2).alias("stddev_price")
    ).collect()[0]

for key, value in price_stats.asDict().items():
    print(f"   {key}: ${value}")

# Price distribution
print("\nPrice Distribution:")
price_dist = prices.withColumn("price", col("price_usd").cast("double")) \
    .filter(col("price").isNotNull()) \
    .withColumn("price_range",
        when(col("price") < 2.50, "Under $2.50")
        .when(col("price") < 5.00, "$2.50 - $4.99")
        .when(col("price") < 10.00, "$5.00 - $9.99")
        .when(col("price") < 20.00, "$10.00 - $19.99")
        .otherwise("$20.00+")
    ) \
    .groupBy("price_range") \
    .agg(count("*").alias("product_count")) \
    .withColumn("percentage", _round((col("product_count") / prices.count()) * 100, 2)) \
    .orderBy(col("product_count").desc())

display(price_dist)

## Orders Table - Detailed Analysis

In [0]:
print("=" * 80)
print("ORDERS TABLE - STRUCTURE AND DATA QUALITY")
print("=" * 80)

orders = spark.table(f"{bronze_schema}.orders")

# Basic structure
print(f"\nTable Structure:")
print(f"   Total Rows: {orders.count():,}")
print(f"   Total Columns: {len(orders.columns)}")

# Schema details
print(f"\nSchema (Data Types):")
orders.printSchema()

# Cardinality of key columns
print(f"\nKey Column Cardinality:")
print(f"   Unique order_id: {orders.select('order_id').distinct().count():,}")
print(f"   Unique user_id: {orders.select('user_id').distinct().count():,}")
print(f"   Unique eval_set values: {orders.select('eval_set').distinct().count()}")

# Check for duplicates
print(f"\nDuplicate Check:")
total_rows = orders.count()
unique_orders = orders.select('order_id').distinct().count()
if total_rows == unique_orders:
    print(f"   No duplicate order_id found (OK)")
else:
    print(f"   WARNING: {total_rows - unique_orders} duplicate order_ids found")

# Value ranges
print(f"\nValue Ranges:")
ranges = orders.agg(
    _min("order_number").alias("min_order_num"),
    _max("order_number").alias("max_order_num"),
    _min("order_dow").alias("min_dow"),
    _max("order_dow").alias("max_dow"),
    _min("order_hour_of_day").alias("min_hour"),
    _max("order_hour_of_day").alias("max_hour")
).collect()[0]

for key, value in ranges.asDict().items():
    print(f"   {key}: {value}")

## Missing Values Analysis

In [0]:
print("=" * 80)
print("MISSING VALUES ANALYSIS")
print("=" * 80)
print("\nAnalyzing NULL values across all tables\n")

for table_name in bronze_tables:
    full_table = f"{bronze_schema}.{table_name}"
    df = spark.table(full_table)
    
    print("\n" + "=" * 80)
    print(f"Table: {table_name.upper()}")
    print("=" * 80)
    
    # Get data columns (exclude metadata)
    data_cols = [c for c in df.columns if c not in ['ingestion_timestamp', 'source_file']]
    
    # Calculate null counts
    null_counts = df.select(
        [_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in data_cols]
    ).collect()[0].asDict()
    
    total_rows = df.count()
    
    # Display results
    has_nulls = False
    null_data = []
    for col_name, null_count in null_counts.items():
        if null_count > 0:
            has_nulls = True
            null_pct = (null_count / total_rows) * 100
            null_data.append((col_name, null_count, null_pct))
    
    if has_nulls:
        print(f"\nColumns with NULL values:")
        print(f"{'Column':<30} {'NULL Count':>15} {'Percentage':>12}")
        print("-" * 80)
        for col_name, null_count, null_pct in sorted(null_data, key=lambda x: x[1], reverse=True):
            print(f"{col_name:<30} {null_count:>15,} {null_pct:>11.2f}%")
    else:
        print("\n   No NULL values detected in any column")
    
    print("\n")